# Chạy Thực Nghiệm Đồ Án (Continual Learning)
Notebook này được thiết kế ĐỘC QUYỀN để chạy toàn bộ file `run_experiments.sh` tự động trên Kaggle.

**HƯỚNG DẪN:**
1. Bật GPU (T4 x2 hoặc P100) trong Settings của Kaggle.
2. Chạy Ô Số 1 để cài đặt môi trường.
3. Bấm **Restart Session / Restart Kernel** khi Kaggle yêu cầu.
4. Chạy Ô Số 2 để bắt đầu quá trình huấn luyện toàn bộ các thực nghiệm (sẽ tốn khoảng 2-3 tiếng, cứ treo máy để đó).

In [1]:
# Ô SỐ 1: CÀI ĐẶT MÔI TRƯỜNG VÀ TẢI MÃ NGUỒN
!git clone https://github.com/quachthanhhmd/bilevel-coresets.git
%cd bilevel-coresets

Cloning into 'bilevel-coresets'...
remote: Enumerating objects: 328, done.
remote: Counting objects: 100% (328/328), done.
remote: Compressing objects: 100% (176/176), done.
remote: Total 328 (delta 161), reused 310 (delta 143), pack-reused 0 (from 0)
Receiving objects: 100% (328/328), 651.16 KiB | 5.38 MiB/s, done.
Resolving deltas: 100% (161/161), done.
/kaggle/working/bilevel-coresets


In [2]:
!git checkout bugfix/fixing-jax-error

Branch 'bugfix/fixing-jax-error' set up to track remote branch 'bugfix/fixing-jax-error' from 'origin'.
Switched to a new branch 'bugfix/fixing-jax-error'


⚠️ **CẢNH BÁO QUAN TRỌNG:** Dừng lại tại đây! Bạn phải bấm `Restart Session` (hoặc `Restart Kernel`) trước khi chạy ô tiếp theo.

In [3]:
# 1. Hạ cấp setuptools để vá lỗi môi trường build của Python 3.12
!pip install "setuptools<70.0.0"

# 2. Cài duy nhất thư viện mô phỏng mạng CNN còn thiếu
!pip install neural-tangents

!pip install --upgrade jax jaxlib==0.1.56+cuda101 -f https://storage.googleapis.com/jax-releases/jax_releases.html

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 894.6/894.6 kB 13.1 MB/s eta 0:00:00 0:00:01
  Attempting uninstall: setuptools
    Found existing installation: setuptools 81.0.0
    Uninstalling setuptools-81.0.0:
      Successfully uninstalled setuptools-81.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.7/248.7 kB 5.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.8/97.8 kB 6.2 MB/s eta 0:00:00
Looking in links: https://storage.googleapis.com/jax-releases/jax_releases.html
ERROR: Ignored the following yanked versions: 0.4.32
ERROR: Could not find a version that satisfies the requirement jaxlib==0.1.56+cuda101 (from versions: 0.4.17, 0.4.18, 0.4.19, 0.4.20, 0.4.21, 0.4.22, 0.4.23, 0.4.24, 0.4.25, 0.4.26, 0.4.27, 0.4.28, 0.4.29, 0.4.30, 0.4.31, 0.4.32, 0.4.33, 0.4.34, 0.4.35, 0.4.36, 0.4.38, 0.5.0, 0.5.1, 0.5.3, 0.6.0, 0.6.1, 0.6.2, 0.7.0, 0.7.1, 0.7.2, 0.8.0, 0.8.1, 0.8.2, 0.8.3, 0.9.0, 0.9.0.1, 0.9.1, 0.9.2, 0.10.0, 0.10.1, 0.10.2, 0.11.0)
ERROR: No matching di

In [4]:
import jax
import jax.core
import jax._src.core
import jax.tree_util
import jax.util

# 1. Vá các class lõi bị giấu (Đã thêm Primitive)
missing_classes = [
    'Jaxpr', 'JaxprEqn', 'Literal', 'Var', 'DropVar', 
    'ClosedJaxpr', 'ShapedArray', 'Value', 'MainTrace', 'Trace',
    'Primitive'  # <--- CHÍNH LÀ THỦ PHẠM MỚI NHẤT
]
for cls_name in missing_classes:
    if hasattr(jax._src.core, cls_name):
        setattr(jax.core, cls_name, getattr(jax._src.core, cls_name))

# 2. Vá hàm tree_multimap
if not hasattr(jax.tree_util, 'tree_multimap'):
    jax.tree_util.tree_multimap = jax.tree_util.tree_map

# 3. Vá hàm safe_map và safe_zip cho jax.util
def custom_safe_map(f, *args):
    return list(map(f, *args))

def custom_safe_zip(*args):
    return list(zip(*args))

jax.util.safe_map = custom_safe_map
jax.util.safe_zip = custom_safe_zip

print("Đã vá nóng JAX V4: Bổ sung Primitive! Sẵn sàng nạp Neural Tangents.")

Đã vá nóng JAX V4: Bổ sung Primitive! Sẵn sàng nạp Neural Tangents.


### Smoke test cho method mới `coreset_nystrom`

`coreset_nystrom` (Table 3) là method **vừa thêm**, dùng CNTK 6 lớp + global average pooling (`bicoreset/cntk.py`) chiếu xuống không gian đặc trưng Nystrom (`bicoreset/nystrom.py`), rồi chọn coreset bằng logistic regression proxy qua `bicoreset.direct.BilevelCoreset`. Mới chỉ được validate bằng kernel giả trong sandbox, **chưa chạy trên jax/GPU thật** -- chạy ô dưới với tham số nhỏ (không phải mặc định) trước để chắc pipeline hoạt động trên môi trường Kaggle của bạn, trước khi thêm `coreset_nystrom` vào danh sách 17 method ở Ô SỐ 2 bên dưới.

In [ ]:
import subprocess, os

repo_root = os.getcwd()
cl_dir = os.path.join(repo_root, 'cl_streaming')
env = os.environ.copy()
env['PYTHONPATH'] = repo_root + os.pathsep + env.get('PYTHONPATH', '')

subprocess.run(['python', 'cl.py',
                '--dataset', 'splitfashionmnist', '--method', 'coreset_nystrom',
                '--samples_per_task', '100', '--buffer_size', '20', '--nr_epochs', '5',
                '--nystrom_dim', '32', '--kernel_batch_size', '4', '--seed', '0'],
               cwd=cl_dir, env=env, check=True)
print('coreset_nystrom smoke test OK -- co the them vao O SO 2.')


In [ ]:
# Ô SỐ 2: HUẤN LUYỆN (nặng, cần GPU) — chỉ ghi ra cl_streaming/cl_results/*.txt, KHÔNG vẽ biểu đồ
# Lưu ý: Lệnh %cd giúp đảm bảo thư mục làm việc luôn nằm trong repo sau khi restart kernel.
%cd /kaggle/working/bilevel-coresets

# Chấp quyền thực thi cho file bash
!chmod +x run_experiments.sh

# Đủ 17 method (13 method Table 3 gốc của paper + sensitivity/glister bạn tự thêm +
# forgetting + coreset_nystrom mới thêm), khớp đúng METHODS trong
# experiments/baseline_comparison_plot.py. coreset_nystrom nặng hơn các method proxy khác
# (phải tính kernel CNTK 6 lớp cho từng task) -- nếu chạy Ô SỐ 2 lần đầu và muốn giảm rủi ro,
# có thể bỏ coreset_nystrom khỏi danh sách dưới và chạy riêng sau khi 16 method còn lại xong.
# -- ô này thay thế cả bản 5-method cũ lẫn ô "remaining_table3_methods" phía dưới,
# không cần chạy 2 lần nữa. run_experiments.sh tự cd vào cl_streaming/ và set PYTHONPATH
# đúng nên không bị lỗi ModuleNotFoundError như khi gọi 'python cl_streaming/cl.py' trực tiếp.
# --stage train: chỉ ghi cl_results/*.txt, không vẽ (vẽ ở Ô "Vẽ lại biểu đồ..." bên dưới).
!./run_experiments.sh --dataset splitfashionmnist \
    --methods uniform,kmeans_features,kmeans_embedding,kmeans_grads,kcenter_features,kcenter_embedding,kcenter_grads,grad_matching,entropy,hardest,frcl,icarl,sensitivity,glister,forgetting,coreset,coreset_nystrom \
    --stage train

🎉 **LẤY KẾT QUẢ:** Sau khi ô số 2 chạy xong (khoảng vài tiếng), kết quả tổng hợp sẽ in ra ngay trên màn hình. Các file kết quả chi tiết `.txt` (chứa dữ liệu dạng JSON) sẽ nằm trong thư mục `cl_streaming/cl_results/`. Bạn có thể tải thư mục này về máy tính.

In [6]:
# %cd /kaggle/working/bilevel-coresets
# !./run_experiments.sh --dataset splitfashionmnist --methods uniform,kmeans_features,kmeans_embedding,kmeans_grads,kcenter_features,kcenter_embedding,kcenter_grads,grad_matching,entropy,hardest,frcl,icarl,sensitivity,glister,forgetting,coreset,coreset_nystrom --stage report

In [7]:
# from IPython.display import Image, display

# for img_name in [
#     'average_forgetting.png',
#     'tradeoff_time_accuracy.png',
#     'baseline_accuracy_comparison.png',
#     'baseline_forgetting_comparison.png',
#     'baseline_tradeoff_comparison.png',
# ]:
#     img_path = f'experiments/{img_name}'
#     print(f'--- {img_name} ---')
#     display(Image(filename=img_path))

## Demo: Binary Logistic Regression coresets

(Đã bỏ phần demo ConvNet-trực-tiếp cho Fashion -- trùng mục đích với pipeline CNTK-Nystrom áp dụng lên FashionMNIST ở phần dưới, chỉ giữ 1 pipeline CNTK để nhất quán với báo cáo.)

`demo_logistic_regression.py` so sánh BiCo (có/không trọng số), k-means++, Sensitivity Coreset cho bài toán logistic regression nhị phân -- chạy RIÊNG cho FashionMNIST (Pullover vs Shirt) và CIFAR-10 (Cat vs Dog), mỗi dataset một chart, không gộp chung. Model là logistic regression trên pixel thô (không phải ConvNet, không phải CNTK/Nystrom), khác cả 2 pipeline kia -- nên giữ lại độc lập.

In [ ]:
# Chạy RIÊNG cho Fashion và CIFAR-10, không gộp chung 1 chart
# (mỗi lệnh tự lưu ra file .png riêng, --output khác nhau)
!python demos/demo_logistic_regression.py --dataset fashion \
    --output demos/logistic_regression_coreset_accuracy_fashion.png
!python demos/demo_logistic_regression.py --dataset cifar10 \
    --output demos/logistic_regression_coreset_accuracy_cifar10.png

In [ ]:
from IPython.display import Image, display

for img_name in [
    'logistic_regression_coreset_accuracy_fashion.png',
    'logistic_regression_coreset_accuracy_cifar10.png',
]:
    print(f'--- {img_name} ---')
    display(Image(filename=f'demos/{img_name}'))


## Continual Learning đầy đủ

Đã gộp vào **Ô SỐ 2** ở đầu notebook — ô đó giờ chạy đủ 17 method (13 method Table 3 gốc của paper + `sensitivity`/`glister` bạn tự thêm + `forgetting`/`coreset_nystrom` mới thêm) cho `splitfashionmnist`, không cần chạy ô riêng nữa.

**Lưu ý:** cần `git pull` (hoặc clone lại) bản mới nhất của nhánh `bugfix/fixing-jax-error` trước khi chạy Ô SỐ 2 — các phương pháp/hàm này chỉ có nếu code đã được cập nhật.

## Vẽ lại biểu đồ với đầy đủ các phương pháp

`baseline_comparison_plot.py` giờ đã biết đủ 17 phương pháp (13 phương pháp Table 3 gốc paper + `sensitivity`/`glister` bạn tự thêm + `forgetting`/`coreset_nystrom`). Phương pháp nào chưa có file kết quả sẽ tự bị bỏ qua (in cảnh báo), không bịa số.

In [ ]:
import subprocess
subprocess.run(['python', 'experiments/baseline_comparison_plot.py',
                '--plot', 'all', '--dataset', 'splitfashionmnist', '--seeds', '0,1,2'], check=True)

from IPython.display import Image, display
for img_name in [
    'baseline_accuracy_comparison.png',
    'baseline_forgetting_comparison.png',
    'baseline_tradeoff_comparison.png',
]:
    print(f'--- {img_name} ---')
    display(Image(filename=f'experiments/{img_name}'))


## Ablation Study

Ablation (buffer size và beta) đã được huấn luyện sẵn trong **Ô số 2** ở trên —
`run_experiments.sh --stage train` luôn chạy kèm khảo sát `buffer_size = 50,100,200`
và `beta = 0,01/1,0/100,0` cho method `coreset`, không phụ thuộc danh sách `--methods`
truyền vào. Không cần lệnh huấn luyện riêng cho ablation.

Ô dưới đây chỉ vẽ lại biểu đồ ablation từ kết quả đã có sẵn trong
`cl_streaming/cl_results/` (đổi `--ablation_method`/`--ablation_seed` nếu muốn xem
method hoặc seed khác).

In [ ]:
import subprocess
subprocess.run(['python', 'experiments/baseline_comparison_plot.py',
                '--plot', 'ablation', '--dataset', 'splitfashionmnist',
                '--ablation_method', 'coreset', '--ablation_seed', '0'], check=True)

from IPython.display import Image, display
display(Image(filename='experiments/coreset_ablation.png'))

## GMM (Gaussian Mixture Model, không giám sát)

Coreset cho bài toán ước lượng mật độ hỗn hợp Gaussian (weighted EM), so sánh BiCo với
Uniform và Sensitivity Coreset qua sai số NLL tương đối. Độc lập với phần Continual
Learning ở trên, không cần GPU. Mỗi dataset mất khoảng 7–8 phút.

In [ ]:
!python experiments/run_gmm_paper.py --dataset fashionmnist --output-dir experiments/gmm_results_fashionmnist

In [ ]:
from IPython.display import Image, display

for dataset in ['fashionmnist', 'kmnist']:
    for img_name in ['gmm_relative_nll_error.png', 'gmm_contours.png']:
        img_path = f'experiments/gmm_results_{dataset}/{img_name}'
        print(f'--- {dataset}: {img_name} ---')
        display(Image(filename=img_path))

## Streaming

So sánh coreset (qua NTK proxy) với reservoir sampling và CBRS trong bối cảnh streaming (không phải continual learning theo task). `streaming.py` giờ đã hỗ trợ `splitfashionmnist`. `reservoir`/`cbrs` không cần jax; `coreset` cần NTK proxy (đã cài ở Ô số 1).

In [ ]:
import subprocess, os

repo_root = os.getcwd()
cl_dir = os.path.join(repo_root, 'cl_streaming')
env = os.environ.copy()
env['PYTHONPATH'] = repo_root + os.pathsep + env.get('PYTHONPATH', '')

for method in ['reservoir', 'cbrs', 'coreset']:
    subprocess.run(['python', 'streaming.py',
                    '--dataset', 'splitfashionmnist', '--method', method,
                    '--seed', '0', '--buffer_size', '100', '--beta', '1.0'],
                   cwd=cl_dir, env=env, check=True)


## Streaming mất cân bằng — Table 6

4 task đầu chỉ giữ 200 điểm, task cuối giữ 2000 điểm (giả lập luồng dữ liệu mất cân bằng). Paper dùng `nr_slots=1` cho thí nghiệm này. `coreset` không được so sánh ở đây (paper chỉ so `reservoir` với `cbrs` cho Table 6).

In [ ]:
import subprocess, os

repo_root = os.getcwd()
cl_dir = os.path.join(repo_root, 'cl_streaming')
env = os.environ.copy()
env['PYTHONPATH'] = repo_root + os.pathsep + env.get('PYTHONPATH', '')

for method in ['reservoir', 'cbrs']:
    subprocess.run(['python', 'streaming.py',
                    '--dataset', 'splitfashionmnistimbalanced', '--method', method,
                    '--seed', '0', '--buffer_size', '100', '--beta', '1.0',
                    '--nr_slots', '1'],
                   cwd=cl_dir, env=env, check=True)


## Xem kết quả Streaming (Table 5 + Table 6)

Chưa có script vẽ biểu đồ riêng cho streaming (paper trình bày dạng bảng số, không phải hình) — ô dưới in trực tiếp từ file JSON.

In [ ]:
import json, os

def load_streaming(dataset, methods, buffer_size=100, beta=1.0, seed=0):
    rows = []
    for method in methods:
        path = f'cl_streaming/streaming_results/{dataset}_{method}_{buffer_size}_{beta}_{seed}.txt'
        if os.path.exists(path):
            with open(path) as f:
                d = json.load(f)
            rows.append((method, d['test_acc'], d['acc_per_task']))
        else:
            rows.append((method, None, None))
    return rows

print('=== Table 5: Streaming (balanced) ===')
for method, acc, per_task in load_streaming('splitfashionmnist', ['reservoir', 'cbrs', 'coreset']):
    print(f'{method:12s} {("%.2f" % acc) if acc is not None else "chưa chạy"}')

print()
print('=== Table 6: Streaming (imbalanced) ===')
for method, acc, per_task in load_streaming('splitfashionmnistimbalanced', ['reservoir', 'cbrs']):
    print(f'{method:12s} {("%.2f" % acc) if acc is not None else "chưa chạy"}')


## Kiểm chứng hằng số chuẩn hóa FashionMNIST (0.2860, 0.3530)

`cl_streaming/datagen.py` dùng `transforms.Normalize((0.2860,), (0.3530,))` cho FashionMNIST. Ô dưới tính lại mean/std thật từ toàn bộ 60000 ảnh train (không phải ước lượng trên mẫu con) và vẽ biểu đồ so sánh — dùng làm bằng chứng trong báo cáo.

In [ ]:
import subprocess
subprocess.run(['python', 'experiments/verify_fashion_normalization.py'], check=True)

from IPython.display import Image, display
display(Image(filename='experiments/fashion_normalization_verification.png'))


---
### Lưu ý: các thực nghiệm CNTK-Nystrom (CIFAR-10/FashionMNIST) và các thực
nghiệm bổ sung còn lại (Dictionary Selection, Batch Active Learning, Deep
network coresets, Joint coresets) không nằm trong notebook này — chạy trực
tiếp các script tương ứng trong `experiments/` nếu cần.
---
